<img src="https://raw.githubusercontent.com/MohammedAly22/qwencleo-asr/main/assets/QwenCleo-ASR-Banner.png" width="100%"/>

# 🎙️ QwenCleo-ASR — FastAPI Server

Egyptian Arabic & code-switching speech recognition, built on Qwen3-ASR.

[Model](https://huggingface.co/mohammedaly22/QwenCleo-ASR) · [GitHub](https://github.com/MohammedAly22/qwencleo-asr) · [PyPI](https://pypi.org/project/qwencleo-asr/)

> **Runtime → Change runtime type → GPU** before running. Then run the cells top to bottom.

Run the QwenCleo FastAPI server inside Colab and call it over HTTP.

## 1. Install (server extras) + get the repo code

In [ ]:
!pip install -q 'qwencleo-asr[server]'
# the server app lives in the GitHub repo
!git clone -q https://github.com/MohammedAly22/qwencleo-asr.git
%cd qwencleo-asr

## 2. Download the model first

Pull the weights now so the server starts almost instantly in the next cell (avoids a connection-refused race while it loads).

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('mohammedaly22/QwenCleo-ASR')
print('model downloaded')

## 3. Launch the server in the background

In [ ]:
import subprocess, os
env = dict(os.environ, QWENCLEO_MODEL='mohammedaly22/QwenCleo-ASR')
proc = subprocess.Popen(
    ['uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    env=env)
print('server starting...')

## 4. Wait until the server is ready

Polls `/health` instead of a fixed sleep, so the cell returns as soon as the server is up (and tells you if it failed).

In [ ]:
import requests, time
for i in range(60):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        print('ready:', r.json()); break
    except Exception:
        if proc.poll() is not None:
            raise RuntimeError('server process exited — check logs above')
        time.sleep(5)
else:
    raise TimeoutError('server did not come up in time')

## 5. Transcribe a file via the API

In [ ]:
# Grab a sample Egyptian/code-switching clip to transcribe
import urllib.request, os
URL = 'https://huggingface.co/mohammedaly22/QwenCleo-ASR/resolve/main/assets/sample.wav'
FALLBACK = 'https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_en.wav'
path = 'sample.wav'
try:
    urllib.request.urlretrieve(URL, path)
except Exception:
    urllib.request.urlretrieve(FALLBACK, path)
print('saved', path, os.path.getsize(path), 'bytes')

In [ ]:
import requests
from IPython.display import display, Audio


audio_path = "sample.wav"
display(Audio(audio_path, autoplay=False))

with open(audio_path, 'rb') as f:
    r = requests.post('http://localhost:8000/v1/transcribe',
                      files={'file': f},
                      data={'language': 'Arabic'})
print(r.json())

## 6. Stop the server

In [ ]:
proc.terminate()